# Task 2: Unsupervised Domain Adaptation on PACS (Step 1: Source-Only ERM)

This notebook sets up the complete, reproducible benchmark protocol for **Tasks 2 & 3** on the **PACS** dataset according to the experimental configuration:
- **Target Domain:** Sketch
- **Source Domains:** Photo, Art Painting, Cartoon (Stratified 80/20 train/val split per source domain, seed `6304`)
- **Model Architecture:** torchvision `ResNet-18` (`ResNet18_Weights.IMAGENET1K_V1`) with a 7-class linear classifier
- **Batch Normalization Policy:** Pretrained ImageNet running mean & variance frozen for all layers; scale ($\gamma$) and bias ($\beta$) remain trainable
- **Balanced Sampling:** 8 samples from each of the 3 source domains ($24$ total) and $24$ unlabeled target samples per iteration
- **Optimization:** AdamW ($lr=10^{-4}$, weight decay=$10^{-4}$), trained up to 30 source epochs, early stopping after 5 non-improving epochs on mean source validation macro-F1
- **Step 1 Focus:** Source-Only Empirical Risk Minimization (ERM) saved as the reference baseline for both Task 2 and Task 3

In [1]:
# Cell 1: Environment Setup & Directory Scaffold with package __init__.py files
import os

REPO_DIRS = [
    "shared/splits",
    "task2/configs",
    "task2/models",
    "task2/methods",
    "task2/evaluation",
    "task2/results/checkpoints",
    "pacs_data"
]

for d in REPO_DIRS:
    os.makedirs(d, exist_ok=True)

# Ensure Python package resolution works across all imports
init_files = [
    "shared/__init__.py",
    "task2/__init__.py",
    "task2/models/__init__.py",
    "task2/methods/__init__.py",
    "task2/evaluation/__init__.py"
]
for f in init_files:
    with open(f, "w") as fp:
        pass

print("Directory structure and package __init__.py files successfully initialized.")

Directory structure and package __init__.py files successfully initialized.


In [2]:
%%writefile shared/pacs_protocol.py
import random
import numpy as np
import torch

SEED = 6304
TARGET_DOMAIN = "sketch"
SOURCE_DOMAINS = ["photo", "art_painting", "cartoon"]
CLASSES = ["dog", "elephant", "giraffe", "guitar", "horse", "house", "person"]
NUM_CLASSES = len(CLASSES)

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


Writing shared/pacs_protocol.py


In [3]:
%%writefile shared/pacs.py
import os
import json
from PIL import Image
import torch
from torch.utils.data import Dataset
from torchvision import transforms
from torchvision.models import ResNet18_Weights
from sklearn.model_selection import StratifiedShuffleSplit
from shared.pacs_protocol import SEED, SOURCE_DOMAINS, TARGET_DOMAIN, CLASSES

weights = ResNet18_Weights.IMAGENET1K_V1
IMAGENET_MEAN = weights.transforms().mean
IMAGENET_STD = weights.transforms().std

TRAIN_TRANSFORM = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

EVAL_TRANSFORM = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

class PACSSubset(Dataset):
    def __init__(self, items, transform=None):
        self.items = items  # list of (path, class_idx, domain_name)
        self.transform = transform

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        path, label, domain = self.items[idx]
        img = Image.open(path).convert("RGB")
        if self.transform is not None:
            img = self.transform(img)
        return img, label, domain

def scan_pacs_directory(root_dir):
    domain_items = {d: [] for d in SOURCE_DOMAINS + [TARGET_DOMAIN]}
    for domain in domain_items.keys():
        dom_path = os.path.join(root_dir, domain)
        if not os.path.exists(dom_path):
            continue
        for class_idx, class_name in enumerate(CLASSES):
            class_path = os.path.join(dom_path, class_name)
            if not os.path.exists(class_path):
                continue
            for fname in sorted(os.listdir(class_path)):
                if fname.lower().endswith((".jpg", ".jpeg", ".png")):
                    fpath = os.path.join(class_path, fname)
                    domain_items[domain].append((fpath, class_idx, domain))
    return domain_items

def build_or_load_splits(root_dir, splits_json_path="shared/splits/pacs_sketch_seed6304.json"):
    if os.path.exists(splits_json_path):
        with open(splits_json_path, "r") as f:
            split_records = json.load(f)
    else:
        domain_items = scan_pacs_directory(root_dir)
        split_records = {"source_train": {}, "source_val": {}, "target_all": []}

        for d in SOURCE_DOMAINS:
            items = domain_items[d]
            labels = [it[1] for it in items]
            # Check if dataset has sufficient class representation for stratification
            test_size = 0.2 if len(items) >= 10 else 0.5
            sss = StratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=SEED)
            train_idx, val_idx = next(sss.split(items, labels))

            split_records["source_train"][d] = [items[i] for i in train_idx]
            split_records["source_val"][d] = [items[i] for i in val_idx]

        split_records["target_all"] = domain_items[TARGET_DOMAIN]
        with open(splits_json_path, "w") as f:
            json.dump(split_records, f, indent=2)

    datasets = {
        "source_train": {
            d: PACSSubset(split_records["source_train"][d], transform=TRAIN_TRANSFORM)
            for d in SOURCE_DOMAINS
        },
        "source_val": {
            d: PACSSubset(split_records["source_val"][d], transform=EVAL_TRANSFORM)
            for d in SOURCE_DOMAINS
        },
        "target_adapt": PACSSubset(split_records["target_all"], transform=TRAIN_TRANSFORM),
        "target_eval": PACSSubset(split_records["target_all"], transform=EVAL_TRANSFORM)
    }
    return datasets


Writing shared/pacs.py


In [4]:
%%writefile task2/models/backbone.py
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights

class ResNet18Backbone(nn.Module):
    def __init__(self):
        super().__init__()
        weights = ResNet18_Weights.IMAGENET1K_V1
        net = resnet18(weights=weights)
        self.features = nn.Sequential(*list(net.children())[:-1])
        self.out_features = 512

    def forward(self, x):
        feat = self.features(x)
        return feat.view(feat.size(0), -1)

def freeze_batchnorm_stats(model):
    """
    Batch-normalization policy: Freeze all running means and variances
    at ImageNet pretrained values while leaving gamma and beta trainable.
    """
    for m in model.modules():
        if isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d)):
            m.eval()
            m.track_running_stats = False


Writing task2/models/backbone.py


In [5]:
%%writefile task2/models/classifier_head.py
import torch.nn as nn
from shared.pacs_protocol import NUM_CLASSES

class LinearClassifierHead(nn.Module):
    def __init__(self, in_features=512, num_classes=NUM_CLASSES):
        super().__init__()
        self.fc = nn.Linear(in_features, num_classes)

    def forward(self, x):
        return self.fc(x)


Writing task2/models/classifier_head.py


In [6]:
%%writefile task2/models/domain_discriminator.py
import torch
import torch.nn as nn
from torch.autograd import Function

class ReverseLayerF(Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.neg() * ctx.alpha, None

class DomainDiscriminator(nn.Module):
    def __init__(self, in_features=512, hidden_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x, alpha=1.0):
        rev = ReverseLayerF.apply(x, alpha)
        return self.net(rev)


Writing task2/models/domain_discriminator.py


In [7]:
%%writefile task2/evaluation/metrics.py
import torch
from sklearn.metrics import accuracy_score, f1_score

@torch.no_grad()
def evaluate_dataset(backbone, classifier, loader, device):
    backbone.eval()
    classifier.eval()
    all_preds, all_targets = [], []

    for imgs, labels, _ in loader:
        imgs = imgs.to(device)
        feats = backbone(imgs)
        logits = classifier(feats)
        preds = logits.argmax(dim=1).cpu()
        all_preds.extend(preds.numpy().tolist())
        all_targets.extend(labels.numpy().tolist())

    acc = accuracy_score(all_targets, all_preds) * 100.0
    macro_f1 = f1_score(all_targets, all_preds, average="macro", zero_division=0) * 100.0
    return acc, macro_f1


Writing task2/evaluation/metrics.py


In [8]:
%%writefile task2/methods/source_only.py
import torch
import torch.nn as nn

class SourceOnlyLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.ce = nn.CrossEntropyLoss()

    def forward(self, source_logits, source_labels, target_features=None):
        loss = self.ce(source_logits, source_labels)
        return loss, {"loss_cls": loss.item(), "loss_transfer": 0.0}


Writing task2/methods/source_only.py


In [9]:
%%writefile task2/train.py
import os
import sys
# Ensure parent directory is in sys.path when executed directly
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))

import copy
import torch
import numpy as np
from torch.utils.data import DataLoader
from shared.pacs_protocol import set_seed, SEED, SOURCE_DOMAINS
from shared.pacs import build_or_load_splits
from task2.models.backbone import ResNet18Backbone, freeze_batchnorm_stats
from task2.models.classifier_head import LinearClassifierHead
from task2.methods.source_only import SourceOnlyLoss
from task2.evaluation.metrics import evaluate_dataset

def run_source_only_erm(data_root="./pacs_data", checkpoint_save_dir="task2/results/checkpoints"):
    set_seed(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Execution device: {device}")

    datasets = build_or_load_splits(data_root)

    # Domain-balanced loader setup: 8 examples per source domain (24 total per iteration)
    source_train_loaders = {
        d: DataLoader(datasets["source_train"][d], batch_size=8, shuffle=True, drop_last=False, num_workers=2)
        for d in SOURCE_DOMAINS
    }
    # Target loader: 24 target examples per iteration
    target_adapt_loader = DataLoader(datasets["target_adapt"], batch_size=24, shuffle=True, drop_last=False, num_workers=2)

    source_val_loaders = {
        d: DataLoader(datasets["source_val"][d], batch_size=64, shuffle=False, num_workers=2)
        for d in SOURCE_DOMAINS
    }
    target_eval_loader = DataLoader(datasets["target_eval"], batch_size=64, shuffle=False, num_workers=2)

    # Instantiate model components
    backbone = ResNet18Backbone().to(device)
    classifier = LinearClassifierHead(in_features=backbone.out_features).to(device)

    criterion = SourceOnlyLoss()
    params = list(backbone.parameters()) + list(classifier.parameters())
    optimizer = torch.optim.AdamW(params, lr=1e-4, weight_decay=1e-4)

    max_source_steps = max(len(loader) for loader in source_train_loaders.values())
    if max_source_steps == 0:
        max_source_steps = 1
    patience = 5
    best_val_macro_f1 = -1.0
    best_weights = None
    no_improve_epochs = 0

    print(f"Starting Source-Only ERM Training (Max 30 source epochs, {max_source_steps} steps/epoch)...")

    for epoch in range(1, 31):
        backbone.train()
        classifier.train()
        freeze_batchnorm_stats(backbone)

        iter_sources = {d: iter(source_train_loaders[d]) for d in SOURCE_DOMAINS}
        iter_target = iter(target_adapt_loader)

        running_loss = 0.0
        for step in range(max_source_steps):
            s_imgs, s_labels = [], []
            for d in SOURCE_DOMAINS:
                try:
                    imgs, labels, _ = next(iter_sources[d])
                except StopIteration:
                    iter_sources[d] = iter(source_train_loaders[d])
                    imgs, labels, _ = next(iter_sources[d])
                s_imgs.append(imgs)
                s_labels.append(labels)

            try:
                t_imgs, _, _ = next(iter_target)
            except StopIteration:
                iter_target = iter(target_adapt_loader)
                t_imgs, _, _ = next(iter_target)

            s_imgs = torch.cat(s_imgs, dim=0).to(device)
            s_labels = torch.cat(s_labels, dim=0).to(device)
            t_imgs = t_imgs.to(device)

            optimizer.zero_grad()
            s_feats = backbone(s_imgs)
            s_logits = classifier(s_feats)

            loss, log_dict = criterion(s_logits, s_labels)
            loss.backward()
            optimizer.step()

            freeze_batchnorm_stats(backbone)
            running_loss += loss.item()

        # Validation across 3 source domains
        val_f1_list = []
        for d in SOURCE_DOMAINS:
            _, val_f1 = evaluate_dataset(backbone, classifier, source_val_loaders[d], device)
            val_f1_list.append(val_f1)

        mean_val_f1 = float(np.mean(val_f1_list))
        epoch_loss = running_loss / max_source_steps
        print(f"Epoch {epoch:02d} | Loss: {epoch_loss:.4f} | Source Val Macro-F1: {mean_val_f1:.2f}% | Per-domain: {[round(x, 2) for x in val_f1_list]}")

        if mean_val_f1 > best_val_macro_f1:
            best_val_macro_f1 = mean_val_f1
            best_weights = {
                "backbone": copy.deepcopy(backbone.state_dict()),
                "classifier": copy.deepcopy(classifier.state_dict()),
                "epoch": epoch,
                "best_val_macro_f1": best_val_macro_f1
            }
            no_improve_epochs = 0
        else:
            no_improve_epochs += 1
            if no_improve_epochs >= patience:
                print(f"Early stopping triggered after {patience} non-improving epochs at epoch {epoch}.")
                break

        # Break condition if synthetic run achieved 100%
        if best_val_macro_f1 >= 100.0 and epoch >= 5:
            break

    checkpoint_path = os.path.join(checkpoint_save_dir, "source_only_erm.pth")
    torch.save(best_weights, checkpoint_path)
    print(f"Best checkpoint saved to: {checkpoint_path} (Validation Macro-F1: {best_val_macro_f1:.2f}%)")

    return checkpoint_path

if __name__ == "__main__":
    run_source_only_erm()


Writing task2/train.py


In [10]:
%%writefile task2/evaluate_final.py
import os
import sys
# Ensure parent directory is in sys.path when executed directly
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))

import torch
import pandas as pd
from torch.utils.data import DataLoader
from shared.pacs_protocol import SOURCE_DOMAINS, TARGET_DOMAIN
from shared.pacs import build_or_load_splits
from task2.models.backbone import ResNet18Backbone
from task2.models.classifier_head import LinearClassifierHead
from task2.evaluation.metrics import evaluate_dataset

def evaluate_source_only_erm(checkpoint_path="task2/results/checkpoints/source_only_erm.pth", data_root="./pacs_data"):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    datasets = build_or_load_splits(data_root)

    backbone = ResNet18Backbone().to(device)
    classifier = LinearClassifierHead(in_features=backbone.out_features).to(device)

    ckpt = torch.load(checkpoint_path, map_location=device)
    backbone.load_state_dict(ckpt["backbone"])
    classifier.load_state_dict(ckpt["classifier"])

    records = []
    # 1. Source validation evaluations
    for d in SOURCE_DOMAINS:
        loader = DataLoader(datasets["source_val"][d], batch_size=64, shuffle=False, num_workers=2)
        acc, f1 = evaluate_dataset(backbone, classifier, loader, device)
        records.append({"Domain": f"{d.capitalize()} (Val)", "Split": "Source Val", "Accuracy (%)": f"{acc:.2f}", "Macro-F1 (%)": f"{f1:.2f}"})

    # 2. Final locked evaluation on unseen Sketch target domain
    target_loader = DataLoader(datasets["target_eval"], batch_size=64, shuffle=False, num_workers=2)
    t_acc, t_f1 = evaluate_dataset(backbone, classifier, target_loader, device)
    records.append({"Domain": f"{TARGET_DOMAIN.capitalize()} (Target)", "Split": "Target Final", "Accuracy (%)": f"{t_acc:.2f}", "Macro-F1 (%)": f"{t_f1:.2f}"})

    df = pd.DataFrame(records)
    csv_path = "task2/results/step1_source_only_erm_summary.csv"
    df.to_csv(csv_path, index=False)
    print("\n--- Source-Only ERM Final Evaluation Report ---")
    print(df.to_string(index=False))
    return df

if __name__ == "__main__":
    evaluate_source_only_erm()


Writing task2/evaluate_final.py


## Synthetic PACS Sample Generator (Self-Contained Verification)

If you do not have the complete PACS dataset downloaded in your current environment, the cell below synthesizes 10 sample images per class for each of the four domains (`photo`, `art_painting`, `cartoon`, `sketch`) into `./pacs_data/` so the complete pipeline, early stopping, and evaluation run seamlessly end-to-end.

In [11]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [12]:
!pip install -q datasets

import os
import shutil
from datasets import load_dataset

PACS_ROOT = "./pacs_data"
DOMAINS = ["photo", "art_painting", "cartoon", "sketch"]
CLASSES = ["dog", "elephant", "giraffe", "guitar", "horse", "house", "person"]

# Clean out any old/synthetic directory
if os.path.exists(PACS_ROOT):
    shutil.rmtree(PACS_ROOT)
os.makedirs(PACS_ROOT, exist_ok=True)

for d in DOMAINS:
    for c in CLASSES:
        os.makedirs(os.path.join(PACS_ROOT, d, c), exist_ok=True)

print("Loading flwrlabs/pacs dataset from Hugging Face...")
# flwrlabs/pacs contains splits corresponding to the domains/partitions
hf_dataset = load_dataset("flwrlabs/pacs")

print("Exporting images to local directory structure (./pacs_data/<domain>/<class>/)...")
# Inspect feature structure and dump to disk
for split_name in hf_dataset.keys():
    split_data = hf_dataset[split_name]
    label_feature = split_data.features.get("label", None)

    for idx, sample in enumerate(split_data):
        img = sample["image"]

        # Domain resolution
        if "domain" in sample:
            domain_val = sample["domain"]
            domain_name = DOMAINS[domain_val] if isinstance(domain_val, int) else str(domain_val).lower()
        else:
            domain_name = split_name.lower()

        # Class label resolution
        label_val = sample["label"]
        if label_feature is not None and hasattr(label_feature, "int2str"):
            class_name = label_feature.int2str(label_val).lower()
        elif isinstance(label_val, int):
            class_name = CLASSES[label_val]
        else:
            class_name = str(label_val).lower()

        # Map to standard canonical naming
        if domain_name in ["art", "artpainting", "art_painting"]:
            domain_name = "art_painting"

        if domain_name in DOMAINS and class_name in CLASSES:
            dest_dir = os.path.join(PACS_ROOT, domain_name, class_name)
            img_path = os.path.join(dest_dir, f"{split_name}_{idx:06d}.jpg")
            img.convert("RGB").save(img_path)

print("\n--- Verifying Exported PACS Dataset ---")
total_count = 0
for d in DOMAINS:
    d_count = sum(len(os.listdir(os.path.join(PACS_ROOT, d, c))) for c in CLASSES)
    print(f"Domain '{d}': {d_count} images")
    total_count += d_count
print(f"Total exported images: {total_count}")

# Remove old split cache so the pipeline constructs the stratified 80/20 split on real images
split_cache = "shared/splits/pacs_sketch_seed6304.json"
if os.path.exists(split_cache):
    os.remove(split_cache)
    print("Cleared previous split cache.")

Loading flwrlabs/pacs dataset from Hugging Face...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  191MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/9991 [00:00<?, ? examples/s]

Exporting images to local directory structure (./pacs_data/<domain>/<class>/)...

--- Verifying Exported PACS Dataset ---
Domain 'photo': 1670 images
Domain 'art_painting': 2048 images
Domain 'cartoon': 2344 images
Domain 'sketch': 3929 images
Total exported images: 9991


In [13]:
# Train Step 1: Source-Only ERM (using module execution syntax to preserve project root path)
!python3 -m task2.train

Execution device: cuda
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100% 44.7M/44.7M [00:00<00:00, 55.2MB/s]
Starting Source-Only ERM Training (Max 30 source epochs, 235 steps/epoch)...
Epoch 01 | Loss: 0.4572 | Source Val Macro-F1: 90.79% | Per-domain: [94.68, 86.93, 90.75]
Epoch 02 | Loss: 0.2149 | Source Val Macro-F1: 91.02% | Per-domain: [94.05, 87.96, 91.04]
Epoch 03 | Loss: 0.1475 | Source Val Macro-F1: 91.60% | Per-domain: [94.11, 89.04, 91.65]
Epoch 04 | Loss: 0.1338 | Source Val Macro-F1: 92.32% | Per-domain: [92.5, 89.49, 94.97]
Epoch 05 | Loss: 0.0766 | Source Val Macro-F1: 89.32% | Per-domain: [93.46, 83.16, 91.35]
Epoch 06 | Loss: 0.0876 | Source Val Macro-F1: 92.77% | Per-domain: [94.64, 89.4, 94.29]
Epoch 07 | Loss: 0.0665 | Source Val Macro-F1: 92.96% | Per-domain: [97.19, 89.5, 92.2]
Epoch 08 | Loss: 0.0715 | Source Val Macro-F1: 90.58% | Per-domain: [93.62, 88.64, 89.47]
Epoch 09 |

In [14]:
# Evaluate Step 1: Validation and Locked Target Evaluation
!python3 -m task2.evaluate_final


--- Source-Only ERM Final Evaluation Report ---
            Domain        Split Accuracy (%) Macro-F1 (%)
       Photo (Val)   Source Val        97.01        96.43
Art_painting (Val)   Source Val        89.76        89.41
     Cartoon (Val)   Source Val        95.95        96.36
   Sketch (Target) Target Final        74.96        76.97


**Part 2: DAN**

In [15]:
%%writefile task2/methods/dan.py
import torch
import torch.nn as nn

class MMDLoss(nn.Module):
    def __init__(self, bandwidth_multipliers=(0.5, 1.0, 2.0)):
        super().__init__()
        self.multipliers = bandwidth_multipliers

    def compute_pairwise_dist_sq(self, x, y):
        """
        Computes ||x_i - y_j||^2 matrix of shape [N, M]
        """
        n = x.size(0)
        m = y.size(0)
        x_norm = (x ** 2).sum(dim=1, keepdim=True).expand(n, m)
        y_norm = (y ** 2).sum(dim=1, keepdim=True).expand(m, n).t()
        dist_sq = x_norm + y_norm - 2.0 * torch.mm(x, y.t())
        return torch.clamp(dist_sq, min=0.0)

    def forward(self, source_features, target_features):
        b_s = source_features.size(0)
        b_t = target_features.size(0)
        combined = torch.cat([source_features, target_features], dim=0)

        # 1. Median heuristic on combined batch
        dist_sq_total = self.compute_pairwise_dist_sq(combined, combined)
        # Extract strictly upper-triangular pairwise distances
        triu_mask = torch.triu(torch.ones_like(dist_sq_total, dtype=torch.bool), diagonal=1)
        pairwise_vals = dist_sq_total[triu_mask]

        median_dist_sq = torch.median(pairwise_vals)
        if median_dist_sq.item() <= 0:
            median_dist_sq = torch.tensor(1.0, device=source_features.device)

        # 2. Pairwise distances for kernel evaluation
        dist_ss = self.compute_pairwise_dist_sq(source_features, source_features)
        dist_tt = self.compute_pairwise_dist_sq(target_features, target_features)
        dist_st = self.compute_pairwise_dist_sq(source_features, target_features)

        # 3. Sum of 3 RBF kernels
        k_ss = torch.zeros_like(dist_ss)
        k_tt = torch.zeros_like(dist_tt)
        k_st = torch.zeros_like(dist_st)

        for mult in self.multipliers:
            bandwidth = 2.0 * (mult * median_dist_sq)
            k_ss = k_ss + torch.exp(-dist_ss / bandwidth)
            k_tt = k_tt + torch.exp(-dist_tt / bandwidth)
            k_st = k_st + torch.exp(-dist_st / bandwidth)

        # 4. Unbiased MMD estimate (excluding self-similar diagonal elements)
        loss_ss = (k_ss.sum() - torch.diagonal(k_ss).sum()) / (b_s * (b_s - 1))
        loss_tt = (k_tt.sum() - torch.diagonal(k_tt).sum()) / (b_t * (b_t - 1))
        loss_st = (2.0 * k_st.sum()) / (b_s * b_t)

        mmd_loss = loss_ss + loss_tt - loss_st
        return torch.clamp(mmd_loss, min=0.0)

class DANLoss(nn.Module):
    def __init__(self, lambda_mmd=1.0):
        super().__init__()
        self.ce = nn.CrossEntropyLoss()
        self.mmd = MMDLoss()
        self.lambda_mmd = lambda_mmd

    def forward(self, source_logits, source_labels, source_features, target_features):
        loss_cls = self.ce(source_logits, source_labels)
        loss_mmd = self.mmd(source_features, target_features)
        total_loss = loss_cls + self.lambda_mmd * loss_mmd
        return total_loss, {
            "loss_cls": loss_cls.item(),
            "loss_mmd": loss_mmd.item(),
            "total_loss": total_loss.item()
        }

Writing task2/methods/dan.py


In [16]:
%%writefile task2/train_dan.py
import os
import sys
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))

import copy
import torch
import numpy as np
from torch.utils.data import DataLoader
from shared.pacs_protocol import set_seed, SEED, SOURCE_DOMAINS
from shared.pacs import build_or_load_splits
from task2.models.backbone import ResNet18Backbone, freeze_batchnorm_stats
from task2.models.classifier_head import LinearClassifierHead
from task2.methods.dan import DANLoss
from task2.evaluation.metrics import evaluate_dataset

def train_dan(data_root="./pacs_data", checkpoint_save_dir="task2/results/checkpoints"):
    set_seed(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Execution device: {device}")

    datasets = build_or_load_splits(data_root)

    # 8 examples per source domain (24 total source per iteration)
    source_train_loaders = {
        d: DataLoader(datasets["source_train"][d], batch_size=8, shuffle=True, drop_last=False, num_workers=2)
        for d in SOURCE_DOMAINS
    }
    # 24 unlabeled target examples per iteration
    target_adapt_loader = DataLoader(datasets["target_adapt"], batch_size=24, shuffle=True, drop_last=False, num_workers=2)

    source_val_loaders = {
        d: DataLoader(datasets["source_val"][d], batch_size=64, shuffle=False, num_workers=2)
        for d in SOURCE_DOMAINS
    }

    # ResNet-18 + Linear Classifier Head
    backbone = ResNet18Backbone().to(device)
    classifier = LinearClassifierHead(in_features=backbone.out_features).to(device)

    criterion = DANLoss(lambda_mmd=1.0)
    params = list(backbone.parameters()) + list(classifier.parameters())
    optimizer = torch.optim.AdamW(params, lr=1e-4, weight_decay=1e-4)

    max_source_steps = max(len(loader) for loader in source_train_loaders.values())
    if max_source_steps == 0:
        max_source_steps = 1
    patience = 5
    best_val_macro_f1 = -1.0
    best_weights = None
    no_improve_epochs = 0

    print(f"Starting DAN Training (Max 30 source epochs, {max_source_steps} steps/epoch, lambda_mmd=1.0)...")

    for epoch in range(1, 31):
        backbone.train()
        classifier.train()
        freeze_batchnorm_stats(backbone)

        iter_sources = {d: iter(source_train_loaders[d]) for d in SOURCE_DOMAINS}
        iter_target = iter(target_adapt_loader)

        running_cls_loss = 0.0
        running_mmd_loss = 0.0

        for step in range(max_source_steps):
            s_imgs, s_labels = [], []
            for d in SOURCE_DOMAINS:
                try:
                    imgs, labels, _ = next(iter_sources[d])
                except StopIteration:
                    iter_sources[d] = iter(source_train_loaders[d])
                    imgs, labels, _ = next(iter_sources[d])
                s_imgs.append(imgs)
                s_labels.append(labels)

            try:
                t_imgs, _, _ = next(iter_target)
            except StopIteration:
                iter_target = iter(target_adapt_loader)
                t_imgs, _, _ = next(iter_target)

            s_imgs = torch.cat(s_imgs, dim=0).to(device)
            s_labels = torch.cat(s_labels, dim=0).to(device)
            t_imgs = t_imgs.to(device)

            optimizer.zero_grad()

            # Extract 512-d features immediately before the linear classifier
            s_feats = backbone(s_imgs)
            t_feats = backbone(t_imgs)
            s_logits = classifier(s_feats)

            loss, log_dict = criterion(s_logits, s_labels, s_feats, t_feats)
            loss.backward()
            optimizer.step()

            freeze_batchnorm_stats(backbone)
            running_cls_loss += log_dict["loss_cls"]
            running_mmd_loss += log_dict["loss_mmd"]

        # Validation across 3 source domains
        val_f1_list = []
        for d in SOURCE_DOMAINS:
            _, val_f1 = evaluate_dataset(backbone, classifier, source_val_loaders[d], device)
            val_f1_list.append(val_f1)

        mean_val_f1 = float(np.mean(val_f1_list))
        avg_cls = running_cls_loss / max_source_steps
        avg_mmd = running_mmd_loss / max_source_steps
        print(f"Epoch {epoch:02d} | Cls Loss: {avg_cls:.4f} | MMD Loss: {avg_mmd:.4f} | Source Val Macro-F1: {mean_val_f1:.2f}% | Per-domain: {[round(x, 2) for x in val_f1_list]}")

        if mean_val_f1 > best_val_macro_f1:
            best_val_macro_f1 = mean_val_f1
            best_weights = {
                "backbone": copy.deepcopy(backbone.state_dict()),
                "classifier": copy.deepcopy(classifier.state_dict()),
                "epoch": epoch,
                "best_val_macro_f1": best_val_macro_f1
            }
            no_improve_epochs = 0
        else:
            no_improve_epochs += 1
            if no_improve_epochs >= patience:
                print(f"Early stopping triggered after {patience} non-improving epochs at epoch {epoch}.")
                break

    os.makedirs(checkpoint_save_dir, exist_ok=True)
    ckpt_path = os.path.join(checkpoint_save_dir, "dan_model.pth")
    torch.save(best_weights, ckpt_path)
    print(f"Best DAN checkpoint saved to: {ckpt_path} (Validation Macro-F1: {best_val_macro_f1:.2f}%)")
    return ckpt_path

if __name__ == "__main__":
    train_dan()

Writing task2/train_dan.py


In [17]:
%%writefile task2/evaluate_dan.py
import os
import sys
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))

import torch
import pandas as pd
from torch.utils.data import DataLoader
from shared.pacs_protocol import SOURCE_DOMAINS, TARGET_DOMAIN
from shared.pacs import build_or_load_splits
from task2.models.backbone import ResNet18Backbone
from task2.models.classifier_head import LinearClassifierHead
from task2.evaluation.metrics import evaluate_dataset

def evaluate_dan(checkpoint_path="task2/results/checkpoints/dan_model.pth", data_root="./pacs_data"):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    datasets = build_or_load_splits(data_root)

    backbone = ResNet18Backbone().to(device)
    classifier = LinearClassifierHead(in_features=backbone.out_features).to(device)

    ckpt = torch.load(checkpoint_path, map_location=device)
    backbone.load_state_dict(ckpt["backbone"])
    classifier.load_state_dict(ckpt["classifier"])

    records = []
    # 1. Source validation evaluations
    for d in SOURCE_DOMAINS:
        loader = DataLoader(datasets["source_val"][d], batch_size=64, shuffle=False, num_workers=2)
        acc, f1 = evaluate_dataset(backbone, classifier, loader, device)
        records.append({"Method": "DAN", "Domain": f"{d.capitalize()} (Val)", "Split": "Source Val", "Accuracy (%)": f"{acc:.2f}", "Macro-F1 (%)": f"{f1:.2f}"})

    # 2. Final locked evaluation on unseen Sketch target domain
    target_loader = DataLoader(datasets["target_eval"], batch_size=64, shuffle=False, num_workers=2)
    t_acc, t_f1 = evaluate_dataset(backbone, classifier, target_loader, device)
    records.append({"Method": "DAN", "Domain": f"{TARGET_DOMAIN.capitalize()} (Target)", "Split": "Target Final", "Accuracy (%)": f"{t_acc:.2f}", "Macro-F1 (%)": f"{t_f1:.2f}"})

    df = pd.DataFrame(records)
    csv_path = "task2/results/step2_dan_summary.csv"
    df.to_csv(csv_path, index=False)
    print("\n--- DAN Final Evaluation Report ---")
    print(df.to_string(index=False))
    return df

if __name__ == "__main__":
    evaluate_dan()

Writing task2/evaluate_dan.py


In [18]:
!python3 -m task2.train_dan

Execution device: cuda
Starting DAN Training (Max 30 source epochs, 235 steps/epoch, lambda_mmd=1.0)...
Epoch 01 | Cls Loss: 0.4488 | MMD Loss: 0.0530 | Source Val Macro-F1: 90.22% | Per-domain: [95.16, 86.4, 89.12]
Epoch 02 | Cls Loss: 0.2171 | MMD Loss: 0.0348 | Source Val Macro-F1: 87.90% | Per-domain: [91.48, 83.76, 88.46]
Epoch 03 | Cls Loss: 0.1327 | MMD Loss: 0.0283 | Source Val Macro-F1: 90.08% | Per-domain: [94.27, 88.15, 87.83]
Epoch 04 | Cls Loss: 0.1120 | MMD Loss: 0.0235 | Source Val Macro-F1: 91.65% | Per-domain: [91.32, 90.4, 93.24]
Epoch 05 | Cls Loss: 0.1035 | MMD Loss: 0.0254 | Source Val Macro-F1: 91.20% | Per-domain: [93.9, 86.9, 92.79]
Epoch 06 | Cls Loss: 0.0719 | MMD Loss: 0.0215 | Source Val Macro-F1: 91.39% | Per-domain: [93.59, 86.24, 94.35]
Epoch 07 | Cls Loss: 0.0595 | MMD Loss: 0.0169 | Source Val Macro-F1: 92.83% | Per-domain: [95.37, 88.77, 94.33]
Epoch 08 | Cls Loss: 0.0481 | MMD Loss: 0.0199 | Source Val Macro-F1: 90.99% | Per-domain: [92.91, 86.35, 93.

In [19]:
!python3 -m task2.evaluate_dan


--- DAN Final Evaluation Report ---
Method             Domain        Split Accuracy (%) Macro-F1 (%)
   DAN        Photo (Val)   Source Val        97.01        96.59
   DAN Art_painting (Val)   Source Val        92.20        92.08
   DAN      Cartoon (Val)   Source Val        94.46        94.95
   DAN    Sketch (Target) Target Final        77.27        73.05


**DANN– Adversarial Alignment**

In [20]:
%%writefile task2/models/domain_discriminator.py
import torch
import torch.nn as nn
from torch.autograd import Function

class ReverseLayerF(Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.neg() * ctx.alpha, None

class DomainDiscriminator(nn.Module):
    def __init__(self, in_features=512, hidden_dim=256, num_domains=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(hidden_dim, num_domains)
        )

    def forward(self, x, alpha=1.0):
        rev_x = ReverseLayerF.apply(x, alpha)
        return self.net(rev_x)

Overwriting task2/models/domain_discriminator.py


In [21]:
%%writefile task2/methods/dann.py
import torch
import torch.nn as nn

class DANNLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.ce_cls = nn.CrossEntropyLoss()
        self.ce_domain = nn.CrossEntropyLoss()

    def forward(self, source_logits, source_labels, source_domain_logits, target_domain_logits):
        loss_cls = self.ce_cls(source_logits, source_labels)

        # Source domain label = 0, Target domain label = 1
        d_source_labels = torch.zeros(source_domain_logits.size(0), dtype=torch.long, device=source_domain_logits.device)
        d_target_labels = torch.ones(target_domain_logits.size(0), dtype=torch.long, device=target_domain_logits.device)

        loss_d_s = self.ce_domain(source_domain_logits, d_source_labels)
        loss_d_t = self.ce_domain(target_domain_logits, d_target_labels)
        loss_domain = 0.5 * (loss_d_s + loss_d_t)

        total_loss = loss_cls + loss_domain
        return total_loss, {
            "loss_cls": loss_cls.item(),
            "loss_domain": loss_domain.item(),
            "total_loss": total_loss.item()
        }

Writing task2/methods/dann.py


In [22]:
%%writefile task2/train_dann.py
import os
import sys
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))

import copy
import numpy as np
import torch
from torch.utils.data import DataLoader
from shared.pacs_protocol import set_seed, SEED, SOURCE_DOMAINS
from shared.pacs import build_or_load_splits
from task2.models.backbone import ResNet18Backbone, freeze_batchnorm_stats
from task2.models.classifier_head import LinearClassifierHead
from task2.models.domain_discriminator import DomainDiscriminator
from task2.methods.dann import DANNLoss
from task2.evaluation.metrics import evaluate_dataset

def compute_alpha(p):
    return float(2.0 / (1.0 + np.exp(-10.0 * p)) - 1.0)

def train_dann(data_root="./pacs_data", checkpoint_save_dir="task2/results/checkpoints"):
    set_seed(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Execution device: {device}")

    datasets = build_or_load_splits(data_root)

    # Balanced source loaders (8 each = 24 source images)
    source_train_loaders = {
        d: DataLoader(datasets["source_train"][d], batch_size=8, shuffle=True, drop_last=False, num_workers=2)
        for d in SOURCE_DOMAINS
    }
    # Unlabeled target loader (24 target images)
    target_adapt_loader = DataLoader(datasets["target_adapt"], batch_size=24, shuffle=True, drop_last=False, num_workers=2)

    source_val_loaders = {
        d: DataLoader(datasets["source_val"][d], batch_size=64, shuffle=False, num_workers=2)
        for d in SOURCE_DOMAINS
    }

    # Model components
    backbone = ResNet18Backbone().to(device)
    classifier = LinearClassifierHead(in_features=backbone.out_features).to(device)
    discriminator = DomainDiscriminator(in_features=backbone.out_features, hidden_dim=256, num_domains=2).to(device)

    criterion = DANNLoss()
    params = list(backbone.parameters()) + list(classifier.parameters()) + list(discriminator.parameters())
    optimizer = torch.optim.AdamW(params, lr=1e-4, weight_decay=1e-4)

    max_source_steps = max(len(loader) for loader in source_train_loaders.values())
    if max_source_steps == 0:
        max_source_steps = 1
    total_epochs = 30
    patience = 5
    best_val_macro_f1 = -1.0
    best_weights = None
    no_improve_epochs = 0

    print(f"Starting DANN Training (Max {total_epochs} epochs, {max_source_steps} steps/epoch)...")

    for epoch in range(1, total_epochs + 1):
        backbone.train()
        classifier.train()
        discriminator.train()
        freeze_batchnorm_stats(backbone)

        iter_sources = {d: iter(source_train_loaders[d]) for d in SOURCE_DOMAINS}
        iter_target = iter(target_adapt_loader)

        running_cls_loss = 0.0
        running_domain_loss = 0.0

        for step in range(max_source_steps):
            # Compute current training progress p in [0, 1] and dynamic alpha
            p = float(epoch - 1 + step / max_source_steps) / float(total_epochs)
            alpha = compute_alpha(p)

            s_imgs, s_labels = [], []
            for d in SOURCE_DOMAINS:
                try:
                    imgs, labels, _ = next(iter_sources[d])
                except StopIteration:
                    iter_sources[d] = iter(source_train_loaders[d])
                    imgs, labels, _ = next(iter_sources[d])
                s_imgs.append(imgs)
                s_labels.append(labels)

            try:
                t_imgs, _, _ = next(iter_target)
            except StopIteration:
                iter_target = iter(target_adapt_loader)
                t_imgs, _, _ = next(iter_target)

            s_imgs = torch.cat(s_imgs, dim=0).to(device)
            s_labels = torch.cat(s_labels, dim=0).to(device)
            t_imgs = t_imgs.to(device)

            optimizer.zero_grad()

            s_feats = backbone(s_imgs)
            t_feats = backbone(t_imgs)
            s_logits = classifier(s_feats)

            s_dom_logits = discriminator(s_feats, alpha=alpha)
            t_dom_logits = discriminator(t_feats, alpha=alpha)

            loss, log_dict = criterion(s_logits, s_labels, s_dom_logits, t_dom_logits)
            loss.backward()
            optimizer.step()

            freeze_batchnorm_stats(backbone)
            running_cls_loss += log_dict["loss_cls"]
            running_domain_loss += log_dict["loss_domain"]

        # Validation across source domains
        val_f1_list = []
        for d in SOURCE_DOMAINS:
            _, val_f1 = evaluate_dataset(backbone, classifier, source_val_loaders[d], device)
            val_f1_list.append(val_f1)

        mean_val_f1 = float(np.mean(val_f1_list))
        avg_cls = running_cls_loss / max_source_steps
        avg_dom = running_domain_loss / max_source_steps
        print(f"Epoch {epoch:02d} | Cls: {avg_cls:.4f} | Domain: {avg_dom:.4f} | alpha: {alpha:.3f} | Source Val Macro-F1: {mean_val_f1:.2f}% | Per-domain: {[round(x, 2) for x in val_f1_list]}")

        if mean_val_f1 > best_val_macro_f1:
            best_val_macro_f1 = mean_val_f1
            best_weights = {
                "backbone": copy.deepcopy(backbone.state_dict()),
                "classifier": copy.deepcopy(classifier.state_dict()),
                "discriminator": copy.deepcopy(discriminator.state_dict()),
                "epoch": epoch,
                "best_val_macro_f1": best_val_macro_f1
            }
            no_improve_epochs = 0
        else:
            no_improve_epochs += 1
            if no_improve_epochs >= patience:
                print(f"Early stopping triggered after {patience} non-improving epochs at epoch {epoch}.")
                break

    os.makedirs(checkpoint_save_dir, exist_ok=True)
    ckpt_path = os.path.join(checkpoint_save_dir, "dann_model.pth")
    torch.save(best_weights, ckpt_path)
    print(f"Best DANN checkpoint saved to: {ckpt_path} (Validation Macro-F1: {best_val_macro_f1:.2f}%)")
    return ckpt_path

if __name__ == "__main__":
    train_dann()

Writing task2/train_dann.py


In [23]:
%%writefile task2/evaluate_dann.py
import os
import sys
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))

import torch
import pandas as pd
from torch.utils.data import DataLoader
from shared.pacs_protocol import SOURCE_DOMAINS, TARGET_DOMAIN
from shared.pacs import build_or_load_splits
from task2.models.backbone import ResNet18Backbone
from task2.models.classifier_head import LinearClassifierHead
from task2.evaluation.metrics import evaluate_dataset

def evaluate_dann(checkpoint_path="task2/results/checkpoints/dann_model.pth", data_root="./pacs_data"):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    datasets = build_or_load_splits(data_root)

    backbone = ResNet18Backbone().to(device)
    classifier = LinearClassifierHead(in_features=backbone.out_features).to(device)

    ckpt = torch.load(checkpoint_path, map_location=device)
    backbone.load_state_dict(ckpt["backbone"])
    classifier.load_state_dict(ckpt["classifier"])

    records = []
    # 1. Source validation evaluations
    for d in SOURCE_DOMAINS:
        loader = DataLoader(datasets["source_val"][d], batch_size=64, shuffle=False, num_workers=2)
        acc, f1 = evaluate_dataset(backbone, classifier, loader, device)
        records.append({"Method": "DANN", "Domain": f"{d.capitalize()} (Val)", "Split": "Source Val", "Accuracy (%)": f"{acc:.2f}", "Macro-F1 (%)": f"{f1:.2f}"})

    # 2. Final locked evaluation on unseen Sketch target domain
    target_loader = DataLoader(datasets["target_eval"], batch_size=64, shuffle=False, num_workers=2)
    t_acc, t_f1 = evaluate_dataset(backbone, classifier, target_loader, device)
    records.append({"Method": "DANN", "Domain": f"{TARGET_DOMAIN.capitalize()} (Target)", "Split": "Target Final", "Accuracy (%)": f"{t_acc:.2f}", "Macro-F1 (%)": f"{t_f1:.2f}"})

    df = pd.DataFrame(records)
    csv_path = "task2/results/step3_dann_summary.csv"
    df.to_csv(csv_path, index=False)
    print("\n--- DANN Final Evaluation Report ---")
    print(df.to_string(index=False))
    return df

if __name__ == "__main__":
    evaluate_dann()

Writing task2/evaluate_dann.py


In [24]:
!python3 -m task2.train_dann

Execution device: cuda
Starting DANN Training (Max 30 epochs, 235 steps/epoch)...
Epoch 01 | Cls: 428.0228 | Domain: 5919.2650 | alpha: 0.164 | Source Val Macro-F1: 6.60% | Per-domain: [7.08, 5.68, 7.04]
Epoch 02 | Cls: 3.8806 | Domain: 35.4873 | alpha: 0.321 | Source Val Macro-F1: 3.32% | Per-domain: [2.78, 3.49, 3.69]
Epoch 03 | Cls: 103.2789 | Domain: 283.4938 | alpha: 0.462 | Source Val Macro-F1: 7.52% | Per-domain: [7.93, 6.56, 8.08]
Epoch 04 | Cls: 150.0842 | Domain: 276.0074 | alpha: 0.582 | Source Val Macro-F1: 2.33% | Per-domain: [2.78, 2.37, 1.85]
Epoch 05 | Cls: 5524728.2257 | Domain: 8072912.9693 | alpha: 0.682 | Source Val Macro-F1: 6.83% | Per-domain: [7.83, 5.64, 7.02]
Epoch 06 | Cls: 13543316.6798 | Domain: 16607422.5618 | alpha: 0.761 | Source Val Macro-F1: 7.67% | Per-domain: [7.32, 5.87, 9.83]
Epoch 07 | Cls: 619277829.1909 | Domain: 579265061.6244 | alpha: 0.823 | Source Val Macro-F1: 4.89% | Per-domain: [4.17, 5.25, 5.24]
Epoch 08 | Cls: 303139733.1753 | Domain: 38

In [25]:
!python3 -m task2.evaluate_dann


--- DANN Final Evaluation Report ---
Method             Domain        Split Accuracy (%) Macro-F1 (%)
  DANN        Photo (Val)   Source Val        26.05         7.32
  DANN Art_painting (Val)   Source Val        22.20         5.87
  DANN      Cartoon (Val)   Source Val        18.98         9.83
  DANN    Sketch (Target) Target Final        18.61         6.17


**CDAN – Class-Conditional Adversarial Alignment**

In [26]:
%%writefile task2/methods/cdan.py
import torch
import torch.nn as nn
import torch.nn.functional as F
from task2.models.domain_discriminator import ReverseLayerF

def multilinear_conditioning(features, logits):
    """
    Computes g(x) = vec(f (x) p), where p = softmax(logits).
    features: [B, D] (512)
    logits: [B, C] (7)
    Returns: [B, D * C] (3584)
    """
    probs = F.softmax(logits, dim=1)
    b, d = features.size()
    c = probs.size(1)
    # Batch outer product
    g = torch.bmm(features.unsqueeze(2), probs.unsqueeze(1))
    return g.view(b, d * c)

class ConditionalDomainDiscriminator(nn.Module):
    def __init__(self, in_features=3584, hidden_dim=256, num_domains=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(hidden_dim, num_domains)
        )

    def forward(self, x, alpha=1.0):
        rev_x = ReverseLayerF.apply(x, alpha)
        return self.net(rev_x)

class CDANLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.ce_cls = nn.CrossEntropyLoss()
        self.ce_domain = nn.CrossEntropyLoss()

    def forward(self, source_logits, source_labels, source_dom_logits, target_dom_logits):
        loss_cls = self.ce_cls(source_logits, source_labels)

        d_source_labels = torch.zeros(source_dom_logits.size(0), dtype=torch.long, device=source_dom_logits.device)
        d_target_labels = torch.ones(target_dom_logits.size(0), dtype=torch.long, device=target_dom_logits.device)

        loss_d_s = self.ce_domain(source_dom_logits, d_source_labels)
        loss_d_t = self.ce_domain(target_dom_logits, d_target_labels)
        loss_domain = 0.5 * (loss_d_s + loss_d_t)

        total_loss = loss_cls + loss_domain
        return total_loss, {
            "loss_cls": loss_cls.item(),
            "loss_domain": loss_domain.item(),
            "total_loss": total_loss.item()
        }

Writing task2/methods/cdan.py


In [27]:
%%writefile task2/train_cdan.py
import os
import sys
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))

import copy
import numpy as np
import torch
from torch.utils.data import DataLoader
from shared.pacs_protocol import set_seed, SEED, SOURCE_DOMAINS, NUM_CLASSES
from shared.pacs import build_or_load_splits
from task2.models.backbone import ResNet18Backbone, freeze_batchnorm_stats
from task2.models.classifier_head import LinearClassifierHead
from task2.methods.cdan import multilinear_conditioning, ConditionalDomainDiscriminator, CDANLoss
from task2.evaluation.metrics import evaluate_dataset

def compute_alpha(p):
    return float(2.0 / (1.0 + np.exp(-10.0 * p)) - 1.0)

def train_cdan(data_root="./pacs_data", checkpoint_save_dir="task2/results/checkpoints"):
    set_seed(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Execution device: {device}")

    datasets = build_or_load_splits(data_root)

    source_train_loaders = {
        d: DataLoader(datasets["source_train"][d], batch_size=8, shuffle=True, drop_last=False, num_workers=2)
        for d in SOURCE_DOMAINS
    }
    target_adapt_loader = DataLoader(datasets["target_adapt"], batch_size=24, shuffle=True, drop_last=False, num_workers=2)

    source_val_loaders = {
        d: DataLoader(datasets["source_val"][d], batch_size=64, shuffle=False, num_workers=2)
        for d in SOURCE_DOMAINS
    }

    # Model architecture
    backbone = ResNet18Backbone().to(device)
    classifier = LinearClassifierHead(in_features=backbone.out_features, num_classes=NUM_CLASSES).to(device)

    # CDAN conditioning dimension: 512 * 7 = 3584
    cond_dim = backbone.out_features * NUM_CLASSES
    discriminator = ConditionalDomainDiscriminator(in_features=cond_dim, hidden_dim=256, num_domains=2).to(device)

    criterion = CDANLoss()
    params = list(backbone.parameters()) + list(classifier.parameters()) + list(discriminator.parameters())
    optimizer = torch.optim.AdamW(params, lr=1e-4, weight_decay=1e-4)

    max_source_steps = max(len(loader) for loader in source_train_loaders.values())
    if max_source_steps == 0:
        max_source_steps = 1
    total_epochs = 30
    patience = 5
    best_val_macro_f1 = -1.0
    best_weights = None
    no_improve_epochs = 0

    print(f"Starting CDAN Training (Max {total_epochs} epochs, {max_source_steps} steps/epoch, Conditioning Dim={cond_dim})...")

    for epoch in range(1, total_epochs + 1):
        backbone.train()
        classifier.train()
        discriminator.train()
        freeze_batchnorm_stats(backbone)

        iter_sources = {d: iter(source_train_loaders[d]) for d in SOURCE_DOMAINS}
        iter_target = iter(target_adapt_loader)

        running_cls_loss = 0.0
        running_dom_loss = 0.0

        for step in range(max_source_steps):
            p = float(epoch - 1 + step / max_source_steps) / float(total_epochs)
            alpha = compute_alpha(p)

            s_imgs, s_labels = [], []
            for d in SOURCE_DOMAINS:
                try:
                    imgs, labels, _ = next(iter_sources[d])
                except StopIteration:
                    iter_sources[d] = iter(source_train_loaders[d])
                    imgs, labels, _ = next(iter_sources[d])
                s_imgs.append(imgs)
                s_labels.append(labels)

            try:
                t_imgs, _, _ = next(iter_target)
            except StopIteration:
                iter_target = iter(target_adapt_loader)
                t_imgs, _, _ = next(iter_target)

            s_imgs = torch.cat(s_imgs, dim=0).to(device)
            s_labels = torch.cat(s_labels, dim=0).to(device)
            t_imgs = t_imgs.to(device)

            optimizer.zero_grad()

            s_feats = backbone(s_imgs)
            t_feats = backbone(t_imgs)

            s_logits = classifier(s_feats)
            t_logits = classifier(t_feats)

            # Class-conditional representations g(x) = vec(f (x) p)
            g_source = multilinear_conditioning(s_feats, s_logits)
            g_target = multilinear_conditioning(t_feats, t_logits)

            s_dom_logits = discriminator(g_source, alpha=alpha)
            t_dom_logits = discriminator(g_target, alpha=alpha)

            loss, log_dict = criterion(s_logits, s_labels, s_dom_logits, t_dom_logits)
            loss.backward()
            optimizer.step()

            freeze_batchnorm_stats(backbone)
            running_cls_loss += log_dict["loss_cls"]
            running_dom_loss += log_dict["loss_domain"]

        # Validation across source domains
        val_f1_list = []
        for d in SOURCE_DOMAINS:
            _, val_f1 = evaluate_dataset(backbone, classifier, source_val_loaders[d], device)
            val_f1_list.append(val_f1)

        mean_val_f1 = float(np.mean(val_f1_list))
        avg_cls = running_cls_loss / max_source_steps
        avg_dom = running_dom_loss / max_source_steps
        print(f"Epoch {epoch:02d} | Cls: {avg_cls:.4f} | Domain: {avg_dom:.4f} | alpha: {alpha:.3f} | Source Val Macro-F1: {mean_val_f1:.2f}% | Per-domain: {[round(x, 2) for x in val_f1_list]}")

        if mean_val_f1 > best_val_macro_f1:
            best_val_macro_f1 = mean_val_f1
            best_weights = {
                "backbone": copy.deepcopy(backbone.state_dict()),
                "classifier": copy.deepcopy(classifier.state_dict()),
                "discriminator": copy.deepcopy(discriminator.state_dict()),
                "epoch": epoch,
                "best_val_macro_f1": best_val_macro_f1
            }
            no_improve_epochs = 0
        else:
            no_improve_epochs += 1
            if no_improve_epochs >= patience:
                print(f"Early stopping triggered after {patience} non-improving epochs at epoch {epoch}.")
                break

    os.makedirs(checkpoint_save_dir, exist_ok=True)
    ckpt_path = os.path.join(checkpoint_save_dir, "cdan_model.pth")
    torch.save(best_weights, ckpt_path)
    print(f"Best CDAN checkpoint saved to: {ckpt_path} (Validation Macro-F1: {best_val_macro_f1:.2f}%)")
    return ckpt_path

if __name__ == "__main__":
    train_cdan()

Writing task2/train_cdan.py


In [28]:
%%writefile task2/evaluate_cdan.py
import os
import sys
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))

import torch
import pandas as pd
from torch.utils.data import DataLoader
from shared.pacs_protocol import SOURCE_DOMAINS, TARGET_DOMAIN
from shared.pacs import build_or_load_splits
from task2.models.backbone import ResNet18Backbone
from task2.models.classifier_head import LinearClassifierHead
from task2.evaluation.metrics import evaluate_dataset

def evaluate_cdan(checkpoint_path="task2/results/checkpoints/cdan_model.pth", data_root="./pacs_data"):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    datasets = build_or_load_splits(data_root)

    backbone = ResNet18Backbone().to(device)
    classifier = LinearClassifierHead(in_features=backbone.out_features).to(device)

    ckpt = torch.load(checkpoint_path, map_location=device)
    backbone.load_state_dict(ckpt["backbone"])
    classifier.load_state_dict(ckpt["classifier"])

    records = []
    for d in SOURCE_DOMAINS:
        loader = DataLoader(datasets["source_val"][d], batch_size=64, shuffle=False, num_workers=2)
        acc, f1 = evaluate_dataset(backbone, classifier, loader, device)
        records.append({"Method": "CDAN", "Domain": f"{d.capitalize()} (Val)", "Split": "Source Val", "Accuracy (%)": f"{acc:.2f}", "Macro-F1 (%)": f"{f1:.2f}"})

    target_loader = DataLoader(datasets["target_eval"], batch_size=64, shuffle=False, num_workers=2)
    t_acc, t_f1 = evaluate_dataset(backbone, classifier, target_loader, device)
    records.append({"Method": "CDAN", "Domain": f"{TARGET_DOMAIN.capitalize()} (Target)", "Split": "Target Final", "Accuracy (%)": f"{t_acc:.2f}", "Macro-F1 (%)": f"{t_f1:.2f}"})

    df = pd.DataFrame(records)
    csv_path = "task2/results/step4_cdan_summary.csv"
    df.to_csv(csv_path, index=False)
    print("\n--- CDAN Final Evaluation Report ---")
    print(df.to_string(index=False))
    return df

if __name__ == "__main__":
    evaluate_cdan()

Writing task2/evaluate_cdan.py


In [29]:
!python3 -m task2.train_cdan

Execution device: cuda
Starting CDAN Training (Max 30 epochs, 235 steps/epoch, Conditioning Dim=3584)...
Epoch 01 | Cls: 0.5540 | Domain: 0.4886 | alpha: 0.164 | Source Val Macro-F1: 89.58% | Per-domain: [92.56, 87.07, 89.1]
Epoch 02 | Cls: 19.8146 | Domain: 256.7673 | alpha: 0.321 | Source Val Macro-F1: 5.51% | Per-domain: [6.37, 5.14, 5.02]
Epoch 03 | Cls: 1.9174 | Domain: 0.6814 | alpha: 0.462 | Source Val Macro-F1: 5.25% | Per-domain: [6.38, 5.14, 4.22]
Epoch 04 | Cls: 1.8849 | Domain: 0.6869 | alpha: 0.582 | Source Val Macro-F1: 14.80% | Per-domain: [17.67, 12.77, 13.95]
Epoch 05 | Cls: 1.8218 | Domain: 0.6857 | alpha: 0.682 | Source Val Macro-F1: 15.48% | Per-domain: [18.6, 15.34, 12.49]
Epoch 06 | Cls: 1.7347 | Domain: 0.6761 | alpha: 0.761 | Source Val Macro-F1: 12.54% | Per-domain: [16.65, 9.57, 11.4]
Early stopping triggered after 5 non-improving epochs at epoch 6.
Best CDAN checkpoint saved to: task2/results/checkpoints/cdan_model.pth (Validation Macro-F1: 89.58%)


In [30]:
!python3 -m task2.evaluate_cdan


--- CDAN Final Evaluation Report ---
Method             Domain        Split Accuracy (%) Macro-F1 (%)
  CDAN        Photo (Val)   Source Val        93.71        92.56
  CDAN Art_painting (Val)   Source Val        87.32        87.07
  CDAN      Cartoon (Val)   Source Val        87.85        89.10
  CDAN    Sketch (Target) Target Final        34.74        36.27


**Common Evaluation and Alignment Diagnostic**

In [31]:
%%writefile task2/evaluation/benchmark_diagnostic.py
import os
import sys
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), "../..")))

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

from shared.pacs_protocol import SEED, SOURCE_DOMAINS, TARGET_DOMAIN, CLASSES
from shared.pacs import build_or_load_splits
from task2.models.backbone import ResNet18Backbone
from task2.models.classifier_head import LinearClassifierHead

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CHECKPOINT_DIR = "task2/results/checkpoints"
RESULTS_DIR = "task2/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

METHODS = {
    "Source-Only": os.path.join(CHECKPOINT_DIR, "source_only_erm.pth"),
    "DAN": os.path.join(CHECKPOINT_DIR, "dan_model.pth"),
    "DANN": os.path.join(CHECKPOINT_DIR, "dann_model.pth"),
    "CDAN": os.path.join(CHECKPOINT_DIR, "cdan_model.pth")
}

@torch.no_grad()
def extract_features_and_preds(backbone, classifier, loader):
    backbone.eval()
    classifier.eval()
    features, logits_list, targets = [], [], []
    for imgs, labels, _ in loader:
        imgs = imgs.to(DEVICE)
        feats = backbone(imgs)
        logits = classifier(feats)
        features.append(feats.cpu())
        logits_list.append(logits.cpu())
        targets.append(labels.cpu())
    features = torch.cat(features, dim=0).numpy()
    logits_list = torch.cat(logits_list, dim=0).numpy()
    targets = torch.cat(targets, dim=0).numpy()
    preds = np.argmax(logits_list, axis=1)
    return features, preds, targets

def compute_domain_separability(source_feats, target_feats):
    # Subsample target features to exactly match source count
    n_source = len(source_feats)
    rng = np.random.RandomState(SEED)
    if len(target_feats) > n_source:
        chosen_indices = rng.choice(len(target_feats), size=n_source, replace=False)
        target_subset = target_feats[chosen_indices]
    else:
        chosen_indices = rng.choice(n_source, size=len(target_feats), replace=False)
        source_feats = source_feats[chosen_indices]
        target_subset = target_feats

    x = np.concatenate([source_feats, target_subset], axis=0)
    y = np.concatenate([np.zeros(len(source_feats)), np.ones(len(target_subset))], axis=0)

    x_train, x_test, y_train, y_test = train_test_split(
        x, y, test_size=0.3, random_state=SEED, stratify=y
    )

    clf = LogisticRegression(C=1.0, max_iter=1000, random_state=SEED)
    clf.fit(x_train, y_train)
    preds = clf.predict(x_test)
    return accuracy_score(y_test, preds) * 100.0

def run_diagnostic():
    print(f"Running Unified Diagnostic across methods on {DEVICE}...")
    datasets = build_or_load_splits("./pacs_data")

    # 1. Prepare data loaders
    source_val_loaders = {
        d: DataLoader(datasets["source_val"][d], batch_size=64, shuffle=False, num_workers=2)
        for d in SOURCE_DOMAINS
    }
    target_loader = DataLoader(datasets["target_eval"], batch_size=64, shuffle=False, num_workers=2)

    summary_records = []
    per_class_records = {}
    confusion_matrices = {}

    baseline_target_acc = None

    for name, ckpt_path in METHODS.items():
        if not os.path.exists(ckpt_path):
            print(f"Skipping {name}: Checkpoint not found at {ckpt_path}")
            continue

        backbone = ResNet18Backbone().to(DEVICE)
        classifier = LinearClassifierHead(in_features=backbone.out_features).to(DEVICE)
        ckpt = torch.load(ckpt_path, map_location=DEVICE)
        backbone.load_state_dict(ckpt["backbone"])
        classifier.load_state_dict(ckpt["classifier"])

        # Evaluate Source Validation
        source_f1s = []
        source_feats_list = []
        for d in SOURCE_DOMAINS:
            f_val, p_val, y_val = extract_features_and_preds(backbone, classifier, source_val_loaders[d])
            source_f1s.append(f1_score(y_val, p_val, average="macro") * 100.0)
            source_feats_list.append(f_val)

        mean_src_val_f1 = float(np.mean(source_f1s))
        pooled_source_feats = np.concatenate(source_feats_list, axis=0)

        # Evaluate Target
        target_feats, target_preds, target_labels = extract_features_and_preds(backbone, classifier, target_loader)
        t_acc = accuracy_score(target_labels, target_preds) * 100.0
        t_f1 = f1_score(target_labels, target_preds, average="macro") * 100.0

        if name == "Source-Only":
            baseline_target_acc = t_acc
            delta_target_acc = 0.0
        else:
            delta_target_acc = t_acc - baseline_target_acc

        # Compute Domain Separability (C=1 Logistic Regression)
        dom_sep = compute_domain_separability(pooled_source_feats, target_feats)

        # Record Per-Class Accuracies
        cm = confusion_matrix(target_labels, target_preds, labels=list(range(len(CLASSES))))
        confusion_matrices[name] = cm
        class_accuracies = cm.diagonal() / cm.sum(axis=1) * 100.0
        per_class_records[name] = class_accuracies

        summary_records.append({
            "Method": name,
            "Source Val Macro-F1 (%)": round(mean_src_val_f1, 2),
            "Target Acc (%)": round(t_acc, 2),
            "Target Macro-F1 (%)": round(t_f1, 2),
            "Δ Target Acc (%)": round(delta_target_acc, 2),
            "Domain Separability (%)": round(dom_sep, 2)
        })

    # Summary Benchmark Table
    df_summary = pd.DataFrame(summary_records)
    summary_path = os.path.join(RESULTS_DIR, "adaptation_summary_benchmark.csv")
    df_summary.to_csv(summary_path, index=False)
    print("\n" + "="*80)
    print("TABLE 1: OVERALL ADAPTATION PERFORMANCE & DOMAIN SEPARABILITY")
    print("="*80)
    print(df_summary.to_string(index=False))

    # Per-Class Accuracy Table
    df_per_class = pd.DataFrame(per_class_records, index=CLASSES)
    df_per_class.to_csv(os.path.join(RESULTS_DIR, "target_per_class_accuracy.csv"))
    print("\n" + "="*80)
    print("TABLE 2: PER-CLASS TARGET ACCURACY (%)")
    print("="*80)
    print(df_per_class.round(2).to_string())

    # Detailed Class Gain/Degradation vs. Source-Only
    if "Source-Only" in per_class_records:
        src_cls = per_class_records["Source-Only"]
        print("\n" + "="*80)
        print("CLASS SHIFT ANALYSIS RELATIVE TO SOURCE-ONLY")
        print("="*80)
        for name in METHODS.keys():
            if name == "Source-Only" or name not in per_class_records:
                continue
            diffs = per_class_records[name] - src_cls
            best_cls = CLASSES[np.argmax(diffs)]
            worst_cls = CLASSES[np.argmin(diffs)]
            print(f"\n[{name}]")
            print(f"  Largest Gain:        {best_cls} (+{diffs[np.argmax(diffs)]:.2f}%)")
            print(f"  Largest Degradation: {worst_cls} ({diffs[np.argmin(diffs)]:.2f}%)")

            # Confusion analysis for worst-performing class
            worst_idx = np.argmin(diffs)
            cm_method = confusion_matrices[name]
            errors = cm_method[worst_idx].copy()
            errors[worst_idx] = 0
            dominant_err_idx = np.argmax(errors)
            print(f"  Dominant Confusion for '{worst_cls}': Misclassified as '{CLASSES[dominant_err_idx]}' ({errors[dominant_err_idx]} samples)")

if __name__ == "__main__":
    run_diagnostic()

Writing task2/evaluation/benchmark_diagnostic.py


In [32]:
!python3 -m task2.evaluation.benchmark_diagnostic

Running Unified Diagnostic across methods on cuda...
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(

TABLE 1: OVERALL ADAPTATION PERFORMANCE & DOMAIN SEPARABILITY
     Method  Source Val Macro-F1 (%)  Target Acc (%)  Target Macro-F1 (%)  Δ Target Acc (%)  Domain Separability (%)
Source-Only                    94.07           74.96                76.97              0.00                    99.59
        DAN                    94.54           77.27                73.05              2.32                    87.91
       DANN           

**Controlled Design Study**

In [33]:
import os
import sys

# Ensure project root is in sys.path
project_root = '/content/'
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# If 'shared' module was previously attempted and failed, it might be cached.
# Remove it from sys.modules to force a fresh import.
if 'shared' in sys.modules:
    del sys.modules['shared']

import copy
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

from shared.pacs_protocol import set_seed, SEED, SOURCE_DOMAINS, TARGET_DOMAIN
from shared.pacs import build_or_load_splits
from task2.models.backbone import ResNet18Backbone, freeze_batchnorm_stats
from task2.models.classifier_head import LinearClassifierHead
from task2.methods.dan import DANLoss
from task2.evaluation.metrics import evaluate_dataset

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RESULTS_DIR = "task2/results/sensitivity_dan"
os.makedirs(RESULTS_DIR, exist_ok=True)

# -------------------------------------------------------------------------
# Helper Functions: Feature Extraction & Domain Separability
# -------------------------------------------------------------------------
@torch.no_grad()
def extract_pooled_features(backbone, loader):
    backbone.eval()
    feats_list = []
    for imgs, _, _ in loader:
        imgs = imgs.to(DEVICE)
        feats_list.append(backbone(imgs).cpu())
    return torch.cat(feats_list, dim=0).numpy()

def compute_domain_separability(source_feats, target_feats):
    n_source = len(source_feats)
    rng = np.random.RandomState(SEED)
    if len(target_feats) > n_source:
        chosen_indices = rng.choice(len(target_feats), size=n_source, replace=False)
        target_subset = target_feats[chosen_indices]
    else:
        chosen_indices = rng.choice(n_source, size=len(target_feats), replace=False)
        source_feats = source_feats[chosen_indices]
        target_subset = target_feats

    x = np.concatenate([source_feats, target_subset], axis=0)
    y = np.concatenate([np.zeros(len(source_feats)), np.ones(len(target_subset))], axis=0)

    x_train, x_test, y_train, y_test = train_test_split(
        x, y, test_size=0.3, random_state=SEED, stratify=y
    )

    clf = LogisticRegression(C=1.0, max_iter=1000, random_state=SEED)
    clf.fit(x_train, y_train)
    return accuracy_score(y_test, clf.predict(x_test)) * 100.0

# -------------------------------------------------------------------------
# Training and Evaluation Function per Lambda Value
# -------------------------------------------------------------------------
def train_and_eval_lambda(lambda_val, datasets):
    set_seed(SEED)
    print(f"\n========================================================")
    print(f"  Training DAN with lambda_MMD = {lambda_val}")
    print(f"========================================================")

    source_train_loaders = {
        d: DataLoader(datasets["source_train"][d], batch_size=8, shuffle=True, drop_last=False, num_workers=2)
        for d in SOURCE_DOMAINS
    }
    target_adapt_loader = DataLoader(datasets["target_adapt"], batch_size=24, shuffle=True, drop_last=False, num_workers=2)

    source_val_loaders = {
        d: DataLoader(datasets["source_val"][d], batch_size=64, shuffle=False, num_workers=2)
        for d in SOURCE_DOMAINS
    }
    target_eval_loader = DataLoader(datasets["target_eval"], batch_size=64, shuffle=False, num_workers=2)

    backbone = ResNet18Backbone().to(DEVICE)
    classifier = LinearClassifierHead(in_features=backbone.out_features).to(DEVICE)

    criterion = DANLoss(lambda_mmd=lambda_val)
    params = list(backbone.parameters()) + list(classifier.parameters())
    optimizer = torch.optim.AdamW(params, lr=1e-4, weight_decay=1e-4)

    max_source_steps = max(len(loader) for loader in source_train_loaders.values())
    patience = 5
    best_val_macro_f1 = -1.0
    best_weights = None
    no_improve_epochs = 0

    for epoch in range(1, 31):
        backbone.train()
        classifier.train()
        freeze_batchnorm_stats(backbone)

        iter_sources = {d: iter(source_train_loaders[d]) for d in SOURCE_DOMAINS}
        iter_target = iter(target_adapt_loader)

        running_cls, running_mmd = 0.0, 0.0
        for step in range(max_source_steps):
            s_imgs, s_labels = [], []
            for d in SOURCE_DOMAINS:
                try:
                    imgs, labels, _ = next(iter_sources[d])
                except StopIteration:
                    iter_sources[d] = iter(source_train_loaders[d])
                    imgs, labels, _ = next(iter_sources[d])
                s_imgs.append(imgs)
                s_labels.append(labels)

            try:
                t_imgs, _, _ = next(iter_target)
            except StopIteration:
                iter_target = iter(target_adapt_loader)
                t_imgs, _, _ = next(iter_target)

            s_imgs = torch.cat(s_imgs, dim=0).to(DEVICE)
            s_labels = torch.cat(s_labels, dim=0).to(DEVICE)
            t_imgs = t_imgs.to(DEVICE)

            optimizer.zero_grad()
            s_feats = backbone(s_imgs)
            t_feats = backbone(t_imgs)
            s_logits = classifier(s_feats)

            loss, log_dict = criterion(s_logits, s_labels, s_feats, t_feats)
            loss.backward()
            optimizer.step()

            freeze_batchnorm_stats(backbone)
            running_cls += log_dict["loss_cls"]
            running_mmd += log_dict["loss_mmd"]

        # Source validation
        val_f1_list = []
        for d in SOURCE_DOMAINS:
            _, val_f1 = evaluate_dataset(backbone, classifier, source_val_loaders[d], DEVICE)
            val_f1_list.append(val_f1)
        mean_val_f1 = float(np.mean(val_f1_list))

        avg_cls = running_cls / max_source_steps
        avg_mmd = running_mmd / max_source_steps
        print(f"Epoch {epoch:02d} | Cls: {avg_cls:.4f} | MMD: {avg_mmd:.4f} | Source Val Macro-F1: {mean_val_f1:.2f}%")

        if mean_val_f1 > best_val_macro_f1:
            best_val_macro_f1 = mean_val_f1
            best_weights = {
                "backbone": copy.deepcopy(backbone.state_dict()),
                "classifier": copy.deepcopy(classifier.state_dict()),
                "epoch": epoch,
                "best_val_macro_f1": best_val_macro_f1
            }
            no_improve_epochs = 0
        else:
            no_improve_epochs += 1
            if no_improve_epochs >= patience:
                print(f"Early stopping at epoch {epoch} (Best Source Val F1: {best_val_macro_f1:.2f}%)")
                break

    # Save checkpoint
    ckpt_path = os.path.join(RESULTS_DIR, f"dan_lambda_{lambda_val}.pth")
    torch.save(best_weights, ckpt_path)

    # Load best checkpoint for final evaluation
    backbone.load_state_dict(best_weights["backbone"])
    classifier.load_state_dict(best_weights["classifier"])

    # Extract source val features for separability score
    source_val_feats = []
    for d in SOURCE_DOMAINS:
        source_val_feats.append(extract_pooled_features(backbone, source_val_loaders[d]))
    pooled_src_feats = np.concatenate(source_val_feats, axis=0)

    # Extract target features and evaluate target
    target_feats = extract_pooled_features(backbone, target_eval_loader)
    t_acc, t_f1 = evaluate_dataset(backbone, classifier, target_eval_loader, DEVICE)

    # Compute domain separability (chance is 50%)
    dom_sep = compute_domain_separability(pooled_src_feats, target_feats)

    return {
        "lambda_MMD": lambda_val,
        "Source Val Macro-F1 (%)": round(best_val_macro_f1, 2),
        "Domain Separability (%)": round(dom_sep, 2),
        "Target Acc (%)": round(t_acc, 2),
        "Target Macro-F1 (%)": round(t_f1, 2)
    }

# -------------------------------------------------------------------------
# Run Study Across All Three Lambdas
# -------------------------------------------------------------------------
datasets = build_or_load_splits("./pacs_data")
lambdas = [0.1, 1.0, 10.0]
records = []

for l in lambdas:
    res = train_and_eval_lambda(l, datasets)
    records.append(res)

df_results = pd.DataFrame(records)
# Compare directly against your locked Source-Only ERM baseline of 74.96%
df_results["Δ Target Acc vs ERM (%)"] = (df_results["Target Acc (%)"] - 74.96).round(2)

# Save and display summary table
out_csv = os.path.join(RESULTS_DIR, "dan_lambda_sensitivity_results.csv")
df_results.to_csv(out_csv, index=False)

print("\n" + "="*85)
print("DAN SENSITIVITY STUDY RESULTS: VARYING lambda_MMD in {0.1, 1.0, 10.0}")
print("="*85)
display(df_results)


  Training DAN with lambda_MMD = 0.1
Epoch 01 | Cls: 0.4709 | MMD: 0.1479 | Source Val Macro-F1: 91.81%
Epoch 02 | Cls: 0.2129 | MMD: 0.0718 | Source Val Macro-F1: 91.17%
Epoch 03 | Cls: 0.1568 | MMD: 0.0693 | Source Val Macro-F1: 90.36%
Epoch 04 | Cls: 0.1209 | MMD: 0.0559 | Source Val Macro-F1: 89.52%
Epoch 05 | Cls: 0.0904 | MMD: 0.0515 | Source Val Macro-F1: 93.09%
Epoch 06 | Cls: 0.0683 | MMD: 0.0432 | Source Val Macro-F1: 94.07%
Epoch 07 | Cls: 0.0884 | MMD: 0.0417 | Source Val Macro-F1: 91.69%
Epoch 08 | Cls: 0.0676 | MMD: 0.0394 | Source Val Macro-F1: 92.19%
Epoch 09 | Cls: 0.0485 | MMD: 0.0402 | Source Val Macro-F1: 90.78%
Epoch 10 | Cls: 0.0565 | MMD: 0.0401 | Source Val Macro-F1: 92.06%
Epoch 11 | Cls: 0.0388 | MMD: 0.0341 | Source Val Macro-F1: 93.60%
Early stopping at epoch 11 (Best Source Val F1: 94.07%)

  Training DAN with lambda_MMD = 1.0
Epoch 01 | Cls: 0.4488 | MMD: 0.0530 | Source Val Macro-F1: 90.22%
Epoch 02 | Cls: 0.2171 | MMD: 0.0348 | Source Val Macro-F1: 87.9

,lambda_MMD,Source Val Macro-F1 (%),Domain Separability (%),Target Acc (%),Target Macro-F1 (%),Δ Target Acc vs ERM (%)
0,0.1,94.07,96.57,70.81,68.76,-4.15
1,1.0,94.54,87.91,77.27,73.05,2.31
2,10.0,91.24,78.30,72.46,65.66,-2.50


In [34]:
import os

os.makedirs("task2/experiments", exist_ok=True)
with open("task2/experiments/__init__.py", "a") as f:
    pass
!python3 -m task2.experiments.dan_sensitivity

/usr/bin/python3: No module named task2.experiments.dan_sensitivity


In [35]:
# Cell 14: Persist Entire Task 1 Directory Structure and Checkpoints to Google Drive
import os
import shutil
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Set source and target paths
source_root = "task2"
target_drive_dir = "/content/drive/MyDrive/ATML_PA1/Task2"

# 3. Create target directory tree on Drive
os.makedirs(target_drive_dir, exist_ok=True)

# 4. Recursively mirror the entire directory structure into Drive
# (copies configs, data, models, analysis, scripts, results, checkpoints, .pt, and .pth files)
shutil.copytree(source_root, target_drive_dir, dirs_exist_ok=True)
print(f"Direct folder sync complete: '{source_root}' -> '{target_drive_dir}'")

# 5. Create an all-inclusive zip archive in the same Drive folder
zip_archive_path = os.path.join(target_drive_dir, "task2_complete_backup")
shutil.make_archive(zip_archive_path, "zip", root_dir=".", base_dir="task2")
print(f"Archive generated at: {zip_archive_path}.zip")

# 6. Verify contents transferred to Drive
print("\nContents saved to Drive:")
for root, dirs, files in os.walk(target_drive_dir):
    level = root.replace(target_drive_dir, "").count(os.sep)
    indent = " " * 4 * level
    print(f"{indent}{os.path.basename(root)}/")
    sub_indent = " " * 4 * (level + 1)
    for f in files:
        print(f"{sub_indent}{f}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Direct folder sync complete: 'task2' -> '/content/drive/MyDrive/ATML_PA1/Task2'
Archive generated at: /content/drive/MyDrive/ATML_PA1/Task2/task2_complete_backup.zip

Contents saved to Drive:
Task2/
    train.py
    evaluate_dan.py
    evaluate_dann.py
    evaluate_cdan.py
    train_cdan.py
    evaluate_final.py
    __init__.py
    train_dan.py
    train_dann.py
    task2_complete_backup.zip
    evaluation/
        benchmark_diagnostic.py
        metrics.py
        __init__.py
        __pycache__/
            __init__.cpython-313.pyc
            benchmark_diagnostic.cpython-313.pyc
            metrics.cpython-313.pyc
    configs/
    models/
        domain_discriminator.py
        backbone.py
        __init__.py
        classifier_head.py
        __pycache__/
            classifier_head.cpython-313.pyc
            __init__.cpython-313.pyc
            domain_d